In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
import stat
import ssl
import subprocess
import sys
import tarfile
import textwrap
import urllib.request
from fractions import Fraction
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path("/")

INPUT_VIDEO: str | None = None

OUTPUT_TAG = "college_volleyball"

HF_TOKEN: str | None = os.environ.get("HF_TOKEN")

SEED = 1234

MAX_VISIBLE_GPUS: int | None = None

MAX_SP_SIZE = 4

CHUNK_KEEP_SECONDS: float | None = 3.5
CHUNK_OVERLAP_SECONDS: float | None = 0.2

CHUNK_PROFILE = "balanced"

CUDA_VISIBLE_DEVICES: str | None = None

DEFAULT_ALLOC_CONF: str | None = "max_split_size_mb:256"

CHUNK_LOSSLESS_PRESET = "ultrafast"

FORCE_REBUILD_ENV = False
FORCE_REDOWNLOAD_REPO = False
FORCE_REDOWNLOAD_WEIGHTS = False
FORCE_RECREATE_MEZZANINE = False
FORCE_RECHUNK = False
FORCE_RERUN_NORMAL = False
FORCE_REASSEMBLE = False

TRACKING_CV_CRF = 8
TRACKING_CV_PRESET = "slow"

DELETE_INTERMEDIATES_AFTER_SUCCESS = True

INPUT_DIR = PROJECT_ROOT / "input"
RUNS_DIR = PROJECT_ROOT / "runs"
BIN_DIR = PROJECT_ROOT / "bin"
REPO_DIR = PROJECT_ROOT / "SeedVR"
ENV_DIR = PROJECT_ROOT / "env_seedvr2"
MICROMAMBA = BIN_DIR / "micromamba"

for p in [PROJECT_ROOT, INPUT_DIR, RUNS_DIR, BIN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"INPUT_DIR     = {INPUT_DIR}")

In [ ]:

VIDEO_EXTS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".ts", ".mts", ".m2ts"}

def run(
    cmd: list[str] | str,
    *,
    cwd: str | Path | None = None,
    env: dict[str, str] | None = None,
    check: bool = True,
    capture: bool = False,
) -> subprocess.CompletedProcess:
    printable = cmd if isinstance(cmd, str) else " ".join(shlex_quote(x) for x in cmd)
    print(f"\n$ {printable}")
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )

def shlex_quote(s: str) -> str:
    import shlex
    return shlex.quote(str(s))

def sanitize_tag(s: str) -> str:
    keep = []
    for ch in s:
        if ch.isalnum() or ch in ("-", "_"):
            keep.append(ch)
        elif ch in (" ", ".", "/"):
            keep.append("_")
    cleaned = "".join(keep).strip("_")
    cleaned = re_sub_many(cleaned)
    return cleaned or "run"

def re_sub_many(s: str) -> str:
    import re
    s = re.sub(r"_+", "_", s)
    return s

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=False))

def read_json(path: Path) -> Any:
    return json.loads(path.read_text())

def resolve_input_video() -> Path:
    if INPUT_VIDEO is not None:
        p = Path(INPUT_VIDEO).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"INPUT_VIDEO does not exist: {p}")
        return p
    candidates = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() in VIDEO_EXTS])
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No input video found. Put a video in {INPUT_DIR} or set INPUT_VIDEO in the config cell."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            f"Found multiple videos in {INPUT_DIR}. Keep exactly one there or set INPUT_VIDEO explicitly.\n"
            + "\n".join(str(p) for p in candidates)
        )
    return candidates[0].resolve()

def list_physical_gpus() -> list[dict[str, Any]]:
    try:
        cp = run(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.total",
                "--format=csv,noheader,nounits",
            ],
            capture=True,
        )
        gpus = []
        for line in cp.stdout.strip().splitlines():
            idx, name, mem = [x.strip() for x in line.split(",", 2)]
            gpus.append({"index": int(idx), "name": name, "memory_total_mb": int(mem)})
        return gpus
    except Exception as e:
        print(f"Warning: failed to query nvidia-smi: {e}")
        return []

def configure_visible_gpus(physical_gpus: list[dict[str, Any]]) -> tuple[str | None, int]:
    global CUDA_VISIBLE_DEVICES
    if CUDA_VISIBLE_DEVICES is not None:
        ids = [x.strip() for x in CUDA_VISIBLE_DEVICES.split(",") if x.strip()]
        return CUDA_VISIBLE_DEVICES, len(ids)
    if MAX_VISIBLE_GPUS is None:
        return None, len(physical_gpus)
    use = max(0, min(MAX_VISIBLE_GPUS, len(physical_gpus)))
    if use == 0:
        return None, 0
    CUDA_VISIBLE_DEVICES = ",".join(str(i) for i in range(use))
    return CUDA_VISIBLE_DEVICES, use

def choose_runtime_topology(visible_gpu_count: int, max_sp_size: int) -> tuple[int, int]:
    if visible_gpu_count <= 0:
        raise RuntimeError("No visible GPUs found.")
    if visible_gpu_count <= max_sp_size:
        return visible_gpu_count, visible_gpu_count if visible_gpu_count > 1 else 1
    groups = visible_gpu_count // max_sp_size
    if groups >= 1:
        return groups * max_sp_size, max_sp_size
    return visible_gpu_count, 1

def fraction_to_float(s: str | float | int | None) -> float:
    if s is None:
        raise ValueError("Cannot convert None to float.")
    if isinstance(s, (float, int)):
        return float(s)
    return float(Fraction(s))

def ffprobe_json(path: Path, ffprobe_bin: Path) -> dict[str, Any]:
    cp = run(
        [
            str(ffprobe_bin),
            "-v", "error",
            "-print_format", "json",
            "-show_streams",
            "-show_format",
            str(path),
        ],
        capture=True,
    )
    return json.loads(cp.stdout)

def count_frames(path: Path, ffprobe_bin: Path) -> int:
    cp = run(
        [
            str(ffprobe_bin),
            "-v", "error",
            "-count_frames",
            "-select_streams", "v:0",
            "-show_entries", "stream=nb_read_frames,nb_frames",
            "-of", "json",
            str(path),
        ],
        capture=True,
    )
    info = json.loads(cp.stdout)
    streams = info.get("streams", [])
    if not streams:
        raise RuntimeError(f"No video stream found in {path}")
    s = streams[0]
    for key in ("nb_read_frames", "nb_frames"):
        if s.get(key) not in (None, "N/A", ""):
            return int(s[key])
    raise RuntimeError(f"Could not determine frame count for {path}")

def choose_chunk_seconds(sp_size: int, min_vram_mb: int | None) -> tuple[float, float]:
    keep = CHUNK_KEEP_SECONDS
    overlap = CHUNK_OVERLAP_SECONDS

    if keep is None:
        profile = str(CHUNK_PROFILE).strip().lower()
        if profile not in {"safe", "balanced", "aggressive"}:
            raise ValueError(f"Unknown CHUNK_PROFILE: {CHUNK_PROFILE}")

        if sp_size >= 4:
            if profile == "safe":
                keep = 1.0
            elif profile == "balanced":
                if min_vram_mb is not None and min_vram_mb >= 120_000:
                    keep = 1.5
                elif min_vram_mb is not None and min_vram_mb >= 80_000:
                    keep = 1.25
                else:
                    keep = 1.0
            else:  # aggressive
                if min_vram_mb is not None and min_vram_mb >= 120_000:
                    keep = 2.0
                elif min_vram_mb is not None and min_vram_mb >= 80_000:
                    keep = 1.5
                else:
                    keep = 1.25
        elif sp_size == 3:
            keep = 0.95 if profile == "aggressive" else (0.85 if profile == "balanced" else 0.70)
        elif sp_size == 2:
            if min_vram_mb is not None and min_vram_mb >= 120_000:
                keep = 0.90 if profile == "aggressive" else (0.75 if profile == "balanced" else 0.60)
            else:
                keep = 0.75 if profile == "aggressive" else (0.65 if profile == "balanced" else 0.55)
        else:
            if min_vram_mb is not None and min_vram_mb >= 120_000:
                keep = 0.70 if profile == "aggressive" else (0.60 if profile == "balanced" else 0.50)
            else:
                keep = 0.60 if profile == "aggressive" else (0.50 if profile == "balanced" else 0.45)

    if overlap is None:
        overlap = 0.20 if sp_size >= 4 else 0.18

    return float(keep), float(overlap)

def build_chunk_manifest(total_frames: int, fps_float: float, keep_seconds: float, overlap_seconds: float) -> dict[str, Any]:
    keep_frames = max(16, int(round(fps_float * keep_seconds)))
    overlap_frames = max(4, int(round(fps_float * overlap_seconds)))
    keep_frames = max(keep_frames, overlap_frames + 8)

    chunks = []
    keep_start = 0
    idx = 0
    while keep_start < total_frames:
        keep_end = min(total_frames, keep_start + keep_frames)
        input_start = max(0, keep_start - overlap_frames)
        input_end = min(total_frames, keep_end + overlap_frames)
        drop_left = keep_start - input_start
        keep_count = keep_end - keep_start
        chunks.append(
            {
                "index": idx,
                "basename": f"chunk_{idx:05d}.mkv",
                "input_start": input_start,
                "input_end": input_end,
                "input_count": input_end - input_start,
                "keep_start": keep_start,
                "keep_end": keep_end,
                "keep_count": keep_count,
                "drop_left": drop_left,
                "drop_right": (input_end - keep_end),
            }
        )
        keep_start = keep_end
        idx += 1

    return {
        "fps_float": fps_float,
        "keep_seconds": keep_seconds,
        "overlap_seconds": overlap_seconds,
        "keep_frames": keep_frames,
        "overlap_frames": overlap_frames,
        "total_frames": total_frames,
        "chunks": chunks,
    }

def env_vars(extra: dict[str, str] | None = None) -> dict[str, str]:
    e = os.environ.copy()
    if CUDA_VISIBLE_DEVICES is not None:
        e["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
    e.setdefault("PYTHONUNBUFFERED", "1")
    e.setdefault("HF_HOME", str(PROJECT_ROOT / "hf_cache"))
    if HF_TOKEN:
        e["HF_TOKEN"] = HF_TOKEN
    e.setdefault("SEEDVR_CHUNK_LOSSLESS_PRESET", str(CHUNK_LOSSLESS_PRESET))
    e.setdefault("TOKENIZERS_PARALLELISM", "false")

    alloc_conf = (
        os.environ.get("PYTORCH_ALLOC_CONF")
        or os.environ.get("PYTORCH_CUDA_ALLOC_CONF")
        or DEFAULT_ALLOC_CONF
    )
    if alloc_conf:
        e["PYTORCH_ALLOC_CONF"] = str(alloc_conf)
        e["PYTORCH_CUDA_ALLOC_CONF"] = str(alloc_conf)
    else:
        e.pop("PYTORCH_ALLOC_CONF", None)
        e.pop("PYTORCH_CUDA_ALLOC_CONF", None)

    if extra:
        e.update({k: str(v) for k, v in extra.items()})
    return e

INPUT_PATH = resolve_input_video()
PHYSICAL_GPUS = list_physical_gpus()
CUDA_VISIBLE_DEVICES, VISIBLE_GPU_COUNT = configure_visible_gpus(PHYSICAL_GPUS)
NPROC_PER_NODE, SP_SIZE = choose_runtime_topology(VISIBLE_GPU_COUNT, MAX_SP_SIZE)
DATA_PARALLEL_GROUPS = max(1, NPROC_PER_NODE // SP_SIZE) if SP_SIZE > 0 else 1

RUN_NAME = f"{sanitize_tag(OUTPUT_TAG)}__{sanitize_tag(INPUT_PATH.stem)}"
RUN_DIR = RUNS_DIR / RUN_NAME
META_DIR = RUN_DIR / "metadata"
MEZZ_DIR = RUN_DIR / "mezzanine"
CHUNK_IN_DIR = RUN_DIR / "chunks" / "input"
CHUNK_NORMAL_DIR = RUN_DIR / "chunks" / "normal_out"
TRIM_NORMAL_DIR = RUN_DIR / "trimmed" / "normal"
FINAL_DIR = RUN_DIR / "final"
TMP_DIR = RUN_DIR / "tmp"
LOG_DIR = RUN_DIR / "logs"

for p in [RUN_DIR, META_DIR, MEZZ_DIR, CHUNK_IN_DIR, CHUNK_NORMAL_DIR, TRIM_NORMAL_DIR, FINAL_DIR, TMP_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"INPUT_PATH           = {INPUT_PATH}")
print(f"RUN_DIR              = {RUN_DIR}")
print(f"CUDA_VISIBLE_DEVICES = {CUDA_VISIBLE_DEVICES}")
print(f"VISIBLE_GPU_COUNT    = {VISIBLE_GPU_COUNT}")
print(f"NPROC_PER_NODE       = {NPROC_PER_NODE}")
print(f"SP_SIZE              = {SP_SIZE}")
print(f"DP_GROUPS            = {DATA_PARALLEL_GROUPS}")
if PHYSICAL_GPUS:
    print("Detected GPUs:")
    for g in PHYSICAL_GPUS:
        print(f"  GPU {g['index']}: {g['name']} ({g['memory_total_mb']/1024:.1f} GB)")
if VISIBLE_GPU_COUNT > NPROC_PER_NODE:
    print(f"Note: using the largest efficient GPU count for SeedVR2 in this run ({NPROC_PER_NODE} of {VISIBLE_GPU_COUNT} visible).")


In [ ]:
def _download_url(url: str, dest: Path) -> None:
    """Download *url* to *dest*, falling back to an unverified SSL context
    when certificate verification fails (expired cert on the server or
    stale CA bundle in the runtime environment)."""
    try:
        urllib.request.urlretrieve(url, dest)
    except urllib.error.URLError as exc:
        if "CERTIFICATE_VERIFY_FAILED" not in str(exc):
            raise
        print("  SSL certificate verification failed — retrying without verification …")
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx))
        with opener.open(url) as resp, open(dest, "wb") as fh:
            shutil.copyfileobj(resp, fh)

def ensure_micromamba() -> None:
    if MICROMAMBA.exists():
        return
    BIN_DIR.mkdir(parents=True, exist_ok=True)
    url = "https://micro.mamba.pm/api/micromamba/linux-64/latest"
    archive_path = PROJECT_ROOT / "micromamba.tar.bz2"
    print(f"Downloading micromamba from {url}")
    _download_url(url, archive_path)
    with tarfile.open(archive_path, "r:bz2") as tar:
        member = tar.getmember("bin/micromamba")
        tar.extract(member, path=PROJECT_ROOT)
    extracted = PROJECT_ROOT / "bin" / "micromamba"
    if extracted != MICROMAMBA:
        shutil.move(str(extracted), str(MICROMAMBA))
    os.chmod(MICROMAMBA, os.stat(MICROMAMBA).st_mode | stat.S_IEXEC)
    archive_path.unlink(missing_ok=True)

def _core_runtime_probe(env_python: Path) -> bool:
    probe = r"""
import importlib
mods = ["torch", "torchvision", "diffusers", "transformers", "flash_attn", "apex", "cv2", "av"]
for m in mods:
    importlib.import_module(m)
print("CORE_RUNTIME_OK")
"""
    cp = run([str(env_python), "-c", probe], env=env_vars(), capture=True, check=False)
    if cp.returncode == 0 and cp.stdout:
        print(cp.stdout)
    return cp.returncode == 0 and "CORE_RUNTIME_OK" in (cp.stdout or "")

def _repair_torchvision_basicsr_compat(env_python: Path) -> None:
    repair = r"""
import sysconfig
from pathlib import Path

site = Path(sysconfig.get_paths()["purelib"])
patched = []

tv_dir = site / "torchvision" / "transforms"
shim = tv_dir / "functional_tensor.py"
if tv_dir.exists() and not shim.exists():
    shim.write_text("from torchvision.transforms.functional import *\n")
    patched.append(str(shim))

bs_dir = site / "basicsr"
if bs_dir.exists():
    for p in bs_dir.rglob("*.py"):
        txt = p.read_text()
        if "torchvision.transforms.functional_tensor" in txt:
            new = txt.replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional")
            if new != txt:
                p.write_text(new)
                patched.append(str(p))

if patched:
    print("Applied torchvision/BasicSR compatibility repairs:")
    for p in patched:
        print(" -", p)
else:
    print("No torchvision/BasicSR compatibility repairs were needed.")
"""
    run([str(env_python), "-c", repair], env=env_vars())

def _verify_runtime(env_python: Path) -> None:
    verify = r"""
import importlib
mods = ["torch", "torchvision", "diffusers", "transformers", "flash_attn", "apex", "cv2", "av"]
for m in mods:
    importlib.import_module(m)

from basicsr.data.degradations import circular_lowpass_kernel, random_mixed_kernels
try:
    from basicsr.utils.img_process_util import filter2D
    print("BasicSR filter2D import ok")
except Exception as e:
    print(f"BasicSR filter2D import skipped: {e}")

import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("device count", torch.cuda.device_count())
print("RUNTIME_VERIFY_OK")
"""
    cp = run([str(env_python), "-c", verify], env=env_vars(), capture=True, check=False)
    if cp.stdout:
        print(cp.stdout)
    if cp.returncode != 0 or "RUNTIME_VERIFY_OK" not in (cp.stdout or ""):
        raise RuntimeError("Runtime verification failed after installation/repair.")

def ensure_env() -> None:
    if FORCE_REBUILD_ENV and ENV_DIR.exists():
        shutil.rmtree(ENV_DIR)
    ensure_micromamba()
    if not ENV_DIR.exists():
        run([
            str(MICROMAMBA),
            "create",
            "-y",
            "-p", str(ENV_DIR),
            "-c", "conda-forge",
            "python=3.10",
            "pip",
            "git",
            "ffmpeg",
            "curl",
            "ca-certificates",
        ], env=env_vars({"MAMBA_ROOT_PREFIX": str(PROJECT_ROOT / ".mamba_root")}))

    env_python = ENV_DIR / "bin" / "python"
    pip_cmd = [str(env_python), "-m", "pip"]

    runtime_ready = env_python.exists() and _core_runtime_probe(env_python)
    if runtime_ready:
        print("Existing SeedVR2 runtime looks healthy; skipping base package re-install.")
    else:
        run(pip_cmd + ["install", "--upgrade", "pip", "setuptools", "wheel", "ninja", "packaging"], env=env_vars())

        run(
            pip_cmd
            + [
                "install",
                "--index-url", "https://download.pytorch.org/whl/cu121",
                "torch==2.4.0",
                "torchvision==0.19.0",
                "torchaudio==2.4.0",
            ],
            env=env_vars(),
        )

        runtime_pkgs = [
            "numpy==1.26.4",
            "einops==0.7.0",
            "omegaconf==2.3.0",
            "opencv-python-headless==4.9.0.80",
            "diffusers==0.29.1",
            "rotary-embedding-torch==0.5.3",
            "transformers==4.38.2",
            "mediapy==1.2.0",
            "basicsr==1.4.2",
            "av==12.0.0",
            "safetensors==0.4.3",
            "sentencepiece==0.2.0",
            "Pillow>=10,<11",
            "tqdm>=4.66,<5",
            "huggingface-hub>=0.23,<1",
            "pyyaml>=6,<7",
            "regex>=2024.0.0",
            "timm>=1.0,<2",
            "accelerate>=0.30,<1",
            "ftfy>=6.2,<7",
            "imageio>=2.34,<3",
        ]
        run(pip_cmd + ["install"] + runtime_pkgs, env=env_vars())

        try:
            run(pip_cmd + ["install", "flash-attn==2.5.9.post1", "--no-build-isolation"], env=env_vars())
        except subprocess.CalledProcessError:
            run(
                pip_cmd + ["install", "flash-attn==2.5.9.post1", "--no-build-isolation", "--no-cache-dir"],
                env=env_vars(),
            )

        apex_url = "https://huggingface.co/ByteDance-Seed/SeedVR2-3B/resolve/main/apex-0.1-cp310-cp310-linux_x86_64.whl"
        run(pip_cmd + ["install", apex_url], env=env_vars())

    _repair_torchvision_basicsr_compat(env_python)

    try:
        _verify_runtime(env_python)
    except RuntimeError:
        print("Trying BasicSR compatibility fallback via basicsr-fixed ...")
        run(
            pip_cmd + ["install", "--upgrade", "--force-reinstall", "--no-deps", "basicsr-fixed"],
            env=env_vars(),
        )
        _repair_torchvision_basicsr_compat(env_python)
        _verify_runtime(env_python)

    ffmpeg_bin = ENV_DIR / "bin" / "ffmpeg"
    cp = run([str(ffmpeg_bin), "-hide_banner", "-encoders"], capture=True, env=env_vars())
    encoders = cp.stdout
    if "libx264" not in encoders or "libx264rgb" not in encoders:
        raise RuntimeError("The ffmpeg build in the runtime environment does not expose libx264/libx264rgb.")
    if "ffv1" not in encoders:
        raise RuntimeError("The ffmpeg build in the runtime environment does not expose ffv1.")

ensure_env()

ENV_PYTHON = ENV_DIR / "bin" / "python"
ENV_FFMPEG = ENV_DIR / "bin" / "ffmpeg"
ENV_FFPROBE = ENV_DIR / "bin" / "ffprobe"
ENV_TORCHRUN = ENV_DIR / "bin" / "torchrun"

print(f"ENV_PYTHON = {ENV_PYTHON}")
print(f"ENV_FFMPEG = {ENV_FFMPEG}")


In [ ]:
def clone_repo() -> None:
    if FORCE_REDOWNLOAD_REPO and REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if not REPO_DIR.exists():
        run([str(ENV_DIR / "bin" / "git"), "clone", "--depth", "1", "https://github.com/ByteDance-Seed/SeedVR.git", str(REPO_DIR)], env=env_vars())

def write_helper_scripts() -> None:
    color_fix_raw = "https://raw.githubusercontent.com/pkuliyi2015/sd-webui-stablesr/master/srmodule/colorfix.py"
    color_fix_path = REPO_DIR / "projects" / "video_diffusion_sr" / "color_fix.py"
    color_fix_path.parent.mkdir(parents=True, exist_ok=True)
    _download_url(color_fix_raw, color_fix_path)

    infer_script_path = REPO_DIR / "projects" / "inference_seedvr2_7b_custom.py"
    infer_script_path.write_text(CUSTOM_INFER_SCRIPT)


def download_weights() -> None:
    ckpt_dir = REPO_DIR / "ckpts"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = ckpt_dir / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    files = [
        ("ByteDance-Seed/SeedVR2-7B", "ema_vae.pth"),
        ("ByteDance-Seed/SeedVR2-7B", "seedvr2_ema_7b.pth"),
    ]

    if not FORCE_REDOWNLOAD_WEIGHTS:
        files = [(repo_id, fname) for repo_id, fname in files if not (ckpt_dir / fname).exists()]

    if not files:
        return

    py = textwrap.dedent(
        f"""
        from huggingface_hub import hf_hub_download
        from pathlib import Path

        cache_dir = Path(r"{cache_dir}")
        local_dir = Path(r"{ckpt_dir}")
        cache_dir.mkdir(parents=True, exist_ok=True)
        local_dir.mkdir(parents=True, exist_ok=True)

        files = {files!r}
        for repo_id, filename in files:
            print(f"Downloading {{repo_id}} / {{filename}}")
            hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                cache_dir=str(cache_dir),
                local_dir=str(local_dir),
                local_dir_use_symlinks=False,
                resume_download=True,
            )
        """
    )
    run([str(ENV_PYTHON), "-c", py], env=env_vars())

CUSTOM_INFER_SCRIPT = r"""
from __future__ import annotations

import argparse
import datetime
import gc
import io
import math
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parents[1]
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
import warnings
from PIL import Image
from einops import rearrange
from omegaconf import OmegaConf
from torchvision.io import read_image
from torchvision.io.video import read_video
from tqdm import tqdm

from common.config import load_config
from common.distributed import get_device, init_torch
from common.distributed.advanced import (
    get_data_parallel_rank,
    get_data_parallel_world_size,
    get_sequence_parallel_rank,
    get_sequence_parallel_world_size,
    init_sequence_parallel,
)
from common.distributed.ops import sync_data
from common.partition import partition_by_groups, partition_by_size
from common.seed import set_seed
from data.image.transforms.na_resize import NaResize
from projects.video_diffusion_sr.infer import VideoDiffusionInfer

warnings.filterwarnings("ignore", message="`torch.cuda.amp.autocast", category=FutureWarning)
warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`", category=FutureWarning)
warnings.filterwarnings("ignore", message="`Transformer2DModelOutput` is deprecated", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*_all_gather_base.*", category=FutureWarning)

try:
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

_real_torch_load = torch.load

def safe_torch_load(*args, **kwargs):
    is_bytes_buffer = bool(args) and isinstance(args[0], io.BytesIO)
    if is_bytes_buffer and "map_location" not in kwargs:
        kwargs = dict(kwargs)
        kwargs["map_location"] = "cpu"
    if "weights_only" not in kwargs:
        kwargs = dict(kwargs)
        kwargs["weights_only"] = False
    try:
        return _real_torch_load(*args, **kwargs)
    except RuntimeError as e:
        msg = str(e).lower()
        if kwargs.get("mmap", False) and ("mmap can only be used" in msg or "mmap is not supported" in msg):
            kwargs = dict(kwargs)
            kwargs.pop("mmap", None)
            return _real_torch_load(*args, **kwargs)
        raise

torch.load = safe_torch_load

if (REPO_ROOT / "projects" / "video_diffusion_sr" / "color_fix.py").exists():
    from projects.video_diffusion_sr.color_fix import wavelet_reconstruction
    USE_COLORFIX = True
else:
    USE_COLORFIX = False
    print("Warning: color_fix.py not found; color fix disabled.")

VIDEO_EXTS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".ts", ".mts", ".m2ts"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

def is_image_file(filename: str) -> bool:
    return Path(filename).suffix.lower() in IMAGE_EXTS

def is_video_or_image_file(filename: str) -> bool:
    return Path(filename).suffix.lower() in VIDEO_EXTS | IMAGE_EXTS

def pad_tchw_to_divisible(video: torch.Tensor, divisor: int = 16) -> tuple[torch.Tensor, tuple[int, int, int, int]]:
    h = int(video.shape[-2])
    w = int(video.shape[-1])
    new_h = int(math.ceil(h / divisor) * divisor)
    new_w = int(math.ceil(w / divisor) * divisor)
    if new_h == h and new_w == w:
        return video, (0, 0, 0, 0)
    pad_top = (new_h - h) // 2
    pad_bottom = new_h - h - pad_top
    pad_left = (new_w - w) // 2
    pad_right = new_w - w - pad_left
    video = F.pad(video, (pad_left, pad_right, pad_top, pad_bottom), mode="replicate")
    return video, (pad_top, pad_bottom, pad_left, pad_right)

def crop_tchw(video: torch.Tensor, pad: tuple[int, int, int, int]) -> torch.Tensor:
    pad_top, pad_bottom, pad_left, pad_right = pad
    h_end = video.shape[-2] - pad_bottom if pad_bottom > 0 else video.shape[-2]
    w_end = video.shape[-1] - pad_right if pad_right > 0 else video.shape[-1]
    return video[..., pad_top:h_end, pad_left:w_end]

def write_lossless_video_x264rgb(output_path: Path, frames_rgb_uint8: np.ndarray, fps: float, ffmpeg_bin: str) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    t, h, w, c = frames_rgb_uint8.shape
    assert c == 3, f"Expected 3 channels, got {c}"
    cmd = [
        ffmpeg_bin,
        "-y",
        "-loglevel", "error",
        "-f", "rawvideo",
        "-pix_fmt", "rgb24",
        "-s:v", f"{w}x{h}",
        "-r", f"{fps:.12f}",
        "-i", "-",
        "-an",
        "-c:v", "libx264rgb",
        "-qp", "0",
        "-preset", os.environ.get("SEEDVR_CHUNK_LOSSLESS_PRESET", "ultrafast"),
        "-g", "1",
        "-bf", "0",
        "-pix_fmt", "rgb24",
        str(output_path),
    ]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    try:
        for i in range(t):
            proc.stdin.write(frames_rgb_uint8[i].tobytes())
    finally:
        if proc.stdin is not None:
            proc.stdin.close()
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"ffmpeg failed while writing {output_path} (exit code {ret})")

def configure_sequence_parallel(sp_size: int) -> None:
    if sp_size > 1:
        init_sequence_parallel(sp_size)

def configure_runner(sp_size: int, checkpoint: str):
    config_path = os.path.join("./configs_7b", "main.yaml")
    config = load_config(config_path)
    runner = VideoDiffusionInfer(config)
    OmegaConf.set_readonly(runner.config, False)

    init_torch(cudnn_benchmark=False, timeout=datetime.timedelta(seconds=3600))
    configure_sequence_parallel(sp_size)

    runner.configure_dit_model(device="cuda", checkpoint=checkpoint)
    runner.configure_vae_model()
    if hasattr(runner.vae, "set_memory_limit"):
        runner.vae.set_memory_limit(**runner.config.vae.memory_limit)
    return runner

def generation_step(runner, text_embeds_dict, cond_latents):
    def _move_to_cuda(x):
        return [i.to(get_device(), non_blocking=False) for i in x]

    cond_latents = [latent.detach().to("cpu", copy=True) if latent.device.type != "cpu" else latent for latent in cond_latents]
    noises = [torch.randn_like(latent, device="cpu") for latent in cond_latents]
    aug_noises = [torch.randn_like(latent, device="cpu") for latent in cond_latents]
    print(f"Generating with noise shape: {noises[0].size()}")

    gc.collect()
    torch.cuda.empty_cache()
    noises, aug_noises, cond_latents = sync_data((noises, aug_noises, cond_latents), 0)
    noises, aug_noises, cond_latents = list(map(lambda x: _move_to_cuda(x), (noises, aug_noises, cond_latents)))

    cond_noise_scale = 0.0

    def _add_noise(x, aug_noise):
        t = torch.tensor([1000.0], device=get_device()) * cond_noise_scale
        shape = torch.tensor(x.shape[1:], device=get_device())[None]
        t = runner.timestep_transform(t, shape)
        print(f"Timestep shifting from {1000.0 * cond_noise_scale} to {t}.")
        return runner.schedule.forward(x, aug_noise, t)

    conditions = [
        runner.get_condition(
            noise,
            task="sr",
            latent_blur=_add_noise(latent_blur, aug_noise),
        )
        for noise, aug_noise, latent_blur in zip(noises, aug_noises, cond_latents)
    ]

    with torch.inference_mode(), torch.autocast("cuda", torch.bfloat16, enabled=True):
        video_tensors = runner.inference(
            noises=noises,
            conditions=conditions,
            dit_offload=True,
            **text_embeds_dict,
        )

    samples = [
        (rearrange(video[:, None], "c t h w -> t c h w") if video.ndim == 3 else rearrange(video, "c t h w -> t c h w"))
        for video in video_tensors
    ]
    del video_tensors
    return samples

def cut_videos(videos: torch.Tensor, sp_size: int) -> torch.Tensor:
    t = int(videos.size(1))
    if t == 1:
        return videos
    if t <= 4 * sp_size:
        padding = [videos[:, -1:].clone()] * (4 * sp_size - t + 1)
        return torch.cat([videos] + padding, dim=1)
    if (t - 1) % (4 * sp_size) == 0:
        return videos
    extra = 4 * sp_size - ((t - 1) % (4 * sp_size))
    padding = [videos[:, -1:].clone()] * extra
    videos = torch.cat([videos] + padding, dim=1)
    assert (videos.size(1) - 1) % (4 * sp_size) == 0
    return videos

def build_inputs(video_path: str, output_dir: str, skip_existing: bool) -> list[str]:
    video_dir = Path(video_path)
    existing = set()
    if skip_existing and Path(output_dir).exists():
        for p in Path(output_dir).iterdir():
            existing.add(p.name)

    items = []
    for p in sorted(video_dir.iterdir()):
        if not p.is_file():
            continue
        if not is_video_or_image_file(p.name):
            continue
        out_name = p.name if p.suffix.lower() in VIDEO_EXTS else p.with_suffix(".png").name
        if skip_existing and out_name in existing:
            continue
        items.append(p.name)
    return items

def generation_loop(
    runner,
    video_path: str,
    output_dir: str,
    checkpoint: str,
    ffmpeg_bin: str,
    batch_size: int = 1,
    cfg_scale: float = 1.0,
    cfg_rescale: float = 0.0,
    sample_steps: int = 1,
    seed: int = 666,
    res_h: int = 720,
    res_w: int = 1280,
    sp_size: int = 1,
    out_fps: float | None = None,
    skip_existing: bool = False,
):
    runner.config.diffusion.cfg.scale = cfg_scale
    runner.config.diffusion.cfg.rescale = cfg_rescale
    runner.config.diffusion.timesteps.sampling.steps = sample_steps
    runner.configure_diffusion()

    set_seed(seed, same_across_ranks=True)
    os.makedirs(output_dir, exist_ok=True)

    original_videos = build_inputs(video_path, output_dir, skip_existing)
    print(f"Total inputs to be generated: {len(original_videos)}")
    if not original_videos:
        return

    original_videos_group = partition_by_groups(
        original_videos,
        get_data_parallel_world_size() // get_sequence_parallel_world_size(),
    )

    original_videos_local = original_videos_group[
        get_data_parallel_rank() // get_sequence_parallel_world_size()
    ]
    original_videos_local = partition_by_size(original_videos_local, batch_size)

    print("Loading prompt embeddings once …")
    text_pos_embeds_cpu = torch.load(REPO_ROOT / "pos_emb.pt", map_location="cpu")
    text_neg_embeds_cpu = torch.load(REPO_ROOT / "neg_emb.pt", map_location="cpu")

    resizer = NaResize(
        resolution=(res_h * res_w) ** 0.5,
        mode="area",
        downsample_only=False,
    )

    for videos in tqdm(original_videos_local, desc="Batches"):
        text_embeds = {
            "texts_pos": [text_pos_embeds_cpu.to(get_device(), non_blocking=True)],
            "texts_neg": [text_neg_embeds_cpu.to(get_device(), non_blocking=True)],
        }
        cond_latents = []
        fps_list = []
        ori_lengths = []
        input_videos = []
        pad_infos = []

        for video_name in videos:
            source_path = Path(video_path) / video_name
            if is_image_file(video_name):
                video = read_image(str(source_path)).unsqueeze(0) / 255.0
                if sp_size > 1:
                    raise ValueError("sp_size must be 1 for image inputs.")
                fps_list.append(1.0 if out_fps is None else out_fps)
            else:
                video, _, info = read_video(str(source_path), output_format="TCHW", pts_unit="sec")
                video = video / 255.0
                fps_list.append(float(info["video_fps"]) if out_fps is None else float(out_fps))

            print(f"Read video size: {tuple(video.size())} from {source_path}")
            ori_lengths.append(int(video.size(0)))

            video = video.to(get_device())
            video = resizer(video)
            video = torch.clamp(video, 0.0, 1.0)
            video, pad_info = pad_tchw_to_divisible(video, divisor=16)
            video = video.sub(0.5).div(0.5)
            video = rearrange(video, "t c h w -> c t h w")

            input_videos.append(video.to("cpu", copy=True))
            pad_infos.append(pad_info)
            cond_latents.append(cut_videos(video, sp_size))

        runner.dit.to("cpu")
        print(f"Encoding videos: {[tuple(x.size()) for x in cond_latents]}")
        runner.vae.to(get_device())
        with torch.inference_mode():
            cond_latents = runner.vae_encode(cond_latents)
        cond_latents = [x.to("cpu", copy=True) for x in cond_latents]
        runner.vae.to("cpu")
        gc.collect()
        torch.cuda.empty_cache()
        runner.dit.to(get_device())

        for i, emb in enumerate(text_embeds["texts_pos"]):
            text_embeds["texts_pos"][i] = emb.to(get_device())
        for i, emb in enumerate(text_embeds["texts_neg"]):
            text_embeds["texts_neg"][i] = emb.to(get_device())

        samples = generation_step(runner, text_embeds, cond_latents=cond_latents)
        runner.dit.to("cpu")
        del cond_latents

        for i, emb in enumerate(text_embeds["texts_pos"]):
            text_embeds["texts_pos"][i] = emb.to("cpu", copy=True)
        for i, emb in enumerate(text_embeds["texts_neg"]):
            text_embeds["texts_neg"][i] = emb.to("cpu", copy=True)
        del text_embeds

        if get_sequence_parallel_rank() == 0:
            for video_name, input_video, sample, ori_length, save_fps, pad_info in zip(
                videos, input_videos, samples, ori_lengths, fps_list, pad_infos
            ):
                sample = sample[:ori_length]

                input_video_tchw = rearrange(input_video, "c t h w -> t c h w")
                if USE_COLORFIX:
                    sample = wavelet_reconstruction(
                        sample.to("cpu"),
                        input_video_tchw[: sample.size(0)].to("cpu"),
                    )
                else:
                    sample = sample.to("cpu")

                sample = crop_tchw(sample, pad_info)
                sample = rearrange(sample, "t c h w -> t h w c")
                sample = sample.clip(-1, 1).mul_(0.5).add_(0.5).mul_(255).round()
                sample = sample.to(torch.uint8).numpy()

                src = Path(video_name)
                if sample.shape[0] == 1 and src.suffix.lower() in IMAGE_EXTS:
                    filename = Path(output_dir) / src.with_suffix(".png").name
                    Image.fromarray(sample.squeeze(0)).save(filename)
                else:
                    filename = Path(output_dir) / src.name
                    write_lossless_video_x264rgb(filename, sample, float(save_fps), ffmpeg_bin)

        gc.collect()
        torch.cuda.empty_cache()

def cleanup_distributed():
    if dist.is_available() and dist.is_initialized():
        try:
            dist.barrier()
        except Exception:
            pass
        try:
            dist.destroy_process_group()
        except Exception:
            pass

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--video_path", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--ffmpeg_bin", type=str, required=True)
    parser.add_argument("--seed", type=int, default=666)
    parser.add_argument("--res_h", type=int, default=720)
    parser.add_argument("--res_w", type=int, default=1280)
    parser.add_argument("--sp_size", type=int, default=1)
    parser.add_argument("--out_fps", type=float, default=None)
    parser.add_argument("--batch_size", type=int, default=1)
    parser.add_argument("--cfg_scale", type=float, default=1.0)
    parser.add_argument("--cfg_rescale", type=float, default=0.0)
    parser.add_argument("--sample_steps", type=int, default=1)
    parser.add_argument("--skip_existing", action="store_true")
    return parser.parse_args()

def main():
    try:
        args = parse_args()
        runner = configure_runner(args.sp_size, args.checkpoint)
        generation_loop(runner, **vars(args))
    finally:
        cleanup_distributed()

if __name__ == "__main__":
    main()
"""

clone_repo()
write_helper_scripts()
download_weights()

verify_script = f"""
import os, sys
repo = r"{REPO_DIR}"
os.chdir(repo)
if repo not in sys.path:
    sys.path.insert(0, repo)
from projects.video_diffusion_sr.infer import VideoDiffusionInfer
print("Repo import check passed.")
"""
run([str(ENV_PYTHON), "-c", verify_script], cwd=REPO_DIR, env=env_vars({"PYTHONPATH": str(REPO_DIR)}))

NORMAL_CKPT = REPO_DIR / "ckpts" / "seedvr2_ema_7b.pth"
VAE_CKPT = REPO_DIR / "ckpts" / "ema_vae.pth"

print(f"NORMAL_CKPT = {NORMAL_CKPT}")


In [ ]:
SOURCE_PROBE = ffprobe_json(INPUT_PATH, ENV_FFPROBE)
VIDEO_STREAMS = [s for s in SOURCE_PROBE["streams"] if s.get("codec_type") == "video"]
AUDIO_STREAMS = [s for s in SOURCE_PROBE["streams"] if s.get("codec_type") == "audio"]
if not VIDEO_STREAMS:
    raise RuntimeError(f"No video stream found in {INPUT_PATH}")
SRC_V = VIDEO_STREAMS[0]

SRC_WIDTH = int(SRC_V["width"])
SRC_HEIGHT = int(SRC_V["height"])
SRC_FPS_STR = SRC_V.get("avg_frame_rate") or SRC_V.get("r_frame_rate")
SRC_FPS_FLOAT = fraction_to_float(SRC_FPS_STR)
HAS_AUDIO = len(AUDIO_STREAMS) > 0

MEZZANINE_PATH = MEZZ_DIR / "source_cfr_lossless.mkv"
MANIFEST_PATH = META_DIR / "chunk_manifest.json"
PARAMS_PATH = META_DIR / "run_params.json"

print(f"Source resolution = {SRC_WIDTH}x{SRC_HEIGHT}")
print(f"Source FPS        = {SRC_FPS_STR} ({SRC_FPS_FLOAT:.6f})")
print(f"Source has audio  = {HAS_AUDIO}")

if FORCE_RECREATE_MEZZANINE or not MEZZANINE_PATH.exists():
    run(
        [
            str(ENV_FFMPEG),
            "-y",
            "-loglevel", "error",
            "-i", str(INPUT_PATH),
            "-map", "0:v:0",
            "-map", "0:a?",
            "-vf", f"fps=fps={SRC_FPS_STR}",
            "-c:v", "ffv1",
            "-level", "3",
            "-coder", "1",
            "-context", "1",
            "-g", "1",
            "-slices", "24",
            "-slicecrc", "1",
            "-c:a", "copy",
            str(MEZZANINE_PATH),
        ],
        env=env_vars(),
    )

MEZZ_FRAMES = count_frames(MEZZANINE_PATH, ENV_FFPROBE)
MIN_VRAM_MB = min([g["memory_total_mb"] for g in PHYSICAL_GPUS], default=None)
KEEP_SEC, OVERLAP_SEC = choose_chunk_seconds(SP_SIZE, MIN_VRAM_MB)

MANIFEST = build_chunk_manifest(
    total_frames=MEZZ_FRAMES,
    fps_float=SRC_FPS_FLOAT,
    keep_seconds=KEEP_SEC,
    overlap_seconds=OVERLAP_SEC,
)
MANIFEST["src_width"] = SRC_WIDTH
MANIFEST["src_height"] = SRC_HEIGHT
MANIFEST["src_fps_str"] = SRC_FPS_STR
MANIFEST["has_audio"] = HAS_AUDIO
MANIFEST["mezzanine"] = str(MEZZANINE_PATH)

write_json(MANIFEST_PATH, MANIFEST)

RUN_PARAMS = {
    "input_video": str(INPUT_PATH),
    "run_name": RUN_NAME,
    "seed": SEED,
    "src_width": SRC_WIDTH,
    "src_height": SRC_HEIGHT,
    "src_fps_str": SRC_FPS_STR,
    "src_fps_float": SRC_FPS_FLOAT,
    "mezzanine_path": str(MEZZANINE_PATH),
    "mezzanine_frames": MEZZ_FRAMES,
    "nproc_per_node": NPROC_PER_NODE,
    "sp_size": SP_SIZE,
    "data_parallel_groups": DATA_PARALLEL_GROUPS,
    "chunk_profile": CHUNK_PROFILE,
    "chunk_keep_seconds": KEEP_SEC,
    "chunk_overlap_seconds": OVERLAP_SEC,
    "chunk_keep_frames": MANIFEST["keep_frames"],
    "chunk_overlap_frames": MANIFEST["overlap_frames"],
    "num_chunks": len(MANIFEST["chunks"]),
    "normal_checkpoint": str(NORMAL_CKPT),
    "final_output": str(FINAL_DIR / "tracking_master_cv.mp4"),
}
write_json(PARAMS_PATH, RUN_PARAMS)

print(json.dumps(RUN_PARAMS, indent=2))

if FORCE_RECHUNK:
    shutil.rmtree(CHUNK_IN_DIR, ignore_errors=True)
    CHUNK_IN_DIR.mkdir(parents=True, exist_ok=True)

for chunk in MANIFEST["chunks"]:
    out_path = CHUNK_IN_DIR / chunk["basename"]
    if out_path.exists() and not FORCE_RECHUNK:
        continue
    vf = (
        f"trim=start_frame={chunk['input_start']}:end_frame={chunk['input_end']},"
        "setpts=PTS-STARTPTS"
    )
    run(
        [
            str(ENV_FFMPEG),
            "-y",
            "-loglevel", "error",
            "-i", str(MEZZANINE_PATH),
            "-vf", vf,
            "-an",
            "-c:v", "ffv1",
            "-level", "3",
            "-coder", "1",
            "-context", "1",
            "-g", "1",
            "-slices", "24",
            "-slicecrc", "1",
            str(out_path),
        ],
        env=env_vars(),
    )

print(f"Created {len(MANIFEST['chunks'])} chunk inputs in {CHUNK_IN_DIR}")


In [ ]:
if FORCE_RERUN_NORMAL:
    shutil.rmtree(CHUNK_NORMAL_DIR, ignore_errors=True)
    CHUNK_NORMAL_DIR.mkdir(parents=True, exist_ok=True)

master_port = str(29500 + random.randint(0, 999))
normal_cmd = [
    str(ENV_TORCHRUN),
    f"--nproc_per_node={NPROC_PER_NODE}",
    f"--master_port={master_port}",
    str(REPO_DIR / "projects" / "inference_seedvr2_7b_custom.py"),
    "--video_path", str(CHUNK_IN_DIR),
    "--output_dir", str(CHUNK_NORMAL_DIR),
    "--checkpoint", str(NORMAL_CKPT),
    "--ffmpeg_bin", str(ENV_FFMPEG),
    "--seed", str(SEED),
    "--res_h", str(SRC_HEIGHT),
    "--res_w", str(SRC_WIDTH),
    "--sp_size", str(SP_SIZE),
    "--sample_steps", "1",
    "--cfg_scale", "1.0",
    "--cfg_rescale", "0.0",
    "--batch_size", "1",
    "--skip_existing",
]

run(
    normal_cmd,
    cwd=REPO_DIR,
    env=env_vars({"PYTHONPATH": str(REPO_DIR)}),
)

print(f"Normal outputs written to {CHUNK_NORMAL_DIR}")

In [ ]:
def remove_path(path: Path) -> None:
    if not path.exists():
        return
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    else:
        path.unlink()

def build_concat_filter_script(manifest: dict[str, Any], script_path: Path) -> None:
    lines = []
    if len(manifest["chunks"]) == 1:
        chunk = manifest["chunks"][0]
        start_frame = int(chunk["drop_left"])
        end_frame = int(chunk["drop_left"] + chunk["keep_count"])
        lines.append(
            f"[0:v]trim=start_frame={start_frame}:end_frame={end_frame},setpts=PTS-STARTPTS[vout]"
        )
    else:
        for idx, chunk in enumerate(manifest["chunks"]):
            start_frame = int(chunk["drop_left"])
            end_frame = int(chunk["drop_left"] + chunk["keep_count"])
            lines.append(
                f"[{idx}:v]trim=start_frame={start_frame}:end_frame={end_frame},setpts=PTS-STARTPTS[v{idx}]"
            )
        concat_inputs = "".join(f"[v{idx}]" for idx in range(len(manifest["chunks"])))
        lines.append(f"{concat_inputs}concat=n={len(manifest['chunks'])}:v=1:a=0[vout]")
    script_path.parent.mkdir(parents=True, exist_ok=True)
    script_path.write_text(";\n".join(lines) + ";\n")

def build_concat_demux_manifest(paths: list[Path], manifest_path: Path) -> None:
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    lines = ["file '" + str(p).replace("'", "'\\''" ) + "'" for p in paths]
    manifest_path.write_text("\n".join(lines) + "\n")

def newest_mtime(paths: list[Path]) -> float:
    times = []
    for p in paths:
        if p.exists():
            times.append(p.stat().st_mtime)
    return max(times) if times else 0.0

def output_is_stale(output_path: Path, inputs: list[Path]) -> bool:
    if not output_path.exists():
        return True
    return output_path.stat().st_mtime < newest_mtime(inputs)

def make_tracking_master_cv_direct(
    manifest: dict[str, Any],
    chunk_dir: Path,
    mezzanine_video: Path,
    tracking_mp4: Path,
    crf: int,
    preset: str,
) -> None:
    filter_script = TMP_DIR / "tracking_master_concat.ffscript"
    build_concat_filter_script(manifest, filter_script)

    inputs = []
    for chunk in manifest["chunks"]:
        src = chunk_dir / chunk["basename"]
        if not src.exists():
            raise FileNotFoundError(f"Missing restored chunk: {src}")
        inputs.extend(["-i", str(src)])

    audio_input_index = len(manifest["chunks"])

    cmd = [
        str(ENV_FFMPEG),
        "-y",
        "-loglevel", "error",
        *inputs,
        "-i", str(mezzanine_video),
        "-filter_complex_script", str(filter_script),
        "-map", "[vout]",
        "-map", f"{audio_input_index}:a?",
        "-c:v", "libx264",
        "-preset", str(preset),
        "-crf", str(crf),
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        str(tracking_mp4),
    ]
    run(cmd, env=env_vars())

def trim_restored_chunks_lossless(manifest: dict[str, Any], chunk_dir: Path, trimmed_dir: Path) -> list[Path]:
    trimmed_dir.mkdir(parents=True, exist_ok=True)
    trimmed_paths: list[Path] = []
    for chunk in manifest["chunks"]:
        src = chunk_dir / chunk["basename"]
        dst = trimmed_dir / chunk["basename"]
        if not src.exists():
            raise FileNotFoundError(f"Missing restored chunk: {src}")

        start_frame = int(chunk["drop_left"])
        end_frame = int(chunk["drop_left"] + chunk["keep_count"])
        cmd = [
            str(ENV_FFMPEG),
            "-y",
            "-loglevel", "error",
            "-i", str(src),
            "-vf", f"trim=start_frame={start_frame}:end_frame={end_frame},setpts=PTS-STARTPTS",
            "-an",
            "-c:v", "ffv1",
            "-level", "3",
            "-coder", "1",
            "-context", "1",
            "-g", "1",
            "-slices", "24",
            "-slicecrc", "1",
            str(dst),
        ]
        run(cmd, env=env_vars())
        trimmed_paths.append(dst)
    return trimmed_paths

def make_tracking_master_cv_exact(
    manifest: dict[str, Any],
    chunk_dir: Path,
    mezzanine_video: Path,
    tracking_mp4: Path,
    crf: int,
    preset: str,
    expected_frames: int,
) -> None:
    exact_dir = TMP_DIR / "exact_trimmed_chunks"
    remove_path(exact_dir)
    exact_dir.mkdir(parents=True, exist_ok=True)
    trimmed_paths = trim_restored_chunks_lossless(manifest, chunk_dir, exact_dir)

    concat_manifest = TMP_DIR / "tracking_master_exact_concat.txt"
    build_concat_demux_manifest(trimmed_paths, concat_manifest)

    cmd = [
        str(ENV_FFMPEG),
        "-y",
        "-loglevel", "error",
        "-f", "concat",
        "-safe", "0",
        "-i", str(concat_manifest),
        "-i", str(mezzanine_video),
        "-map", "0:v:0",
        "-map", "1:a?",
        "-frames:v", str(expected_frames),
        "-c:v", "libx264",
        "-preset", str(preset),
        "-crf", str(crf),
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        str(tracking_mp4),
    ]
    run(cmd, env=env_vars())

TRACKING_MASTER_CV = FINAL_DIR / "tracking_master_cv.mp4"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
for p in list(FINAL_DIR.iterdir()):
    if p.name != TRACKING_MASTER_CV.name:
        remove_path(p)

manifest = read_json(MANIFEST_PATH)
expected_frames = MEZZ_FRAMES

chunk_paths = [CHUNK_NORMAL_DIR / chunk["basename"] for chunk in manifest["chunks"]]
reassembly_inputs = [MEZZANINE_PATH, MANIFEST_PATH, *chunk_paths]
should_reassemble = FORCE_REASSEMBLE or output_is_stale(TRACKING_MASTER_CV, reassembly_inputs)

if should_reassemble:
    make_tracking_master_cv_direct(
        manifest=manifest,
        chunk_dir=CHUNK_NORMAL_DIR,
        mezzanine_video=MEZZANINE_PATH,
        tracking_mp4=TRACKING_MASTER_CV,
        crf=TRACKING_CV_CRF,
        preset=TRACKING_CV_PRESET,
    )
else:
    print("tracking_master_cv.mp4 is newer than the manifest and restored chunks; skipping rebuild.")

tracking_frames = count_frames(TRACKING_MASTER_CV, ENV_FFPROBE)

if tracking_frames != expected_frames:
    print(
        f"Fast reassembly produced {tracking_frames} frames; expected {expected_frames}. "
        "Retrying with the exact lossless trim/concat path ..."
    )
    make_tracking_master_cv_exact(
        manifest=manifest,
        chunk_dir=CHUNK_NORMAL_DIR,
        mezzanine_video=MEZZANINE_PATH,
        tracking_mp4=TRACKING_MASTER_CV,
        crf=TRACKING_CV_CRF,
        preset=TRACKING_CV_PRESET,
        expected_frames=expected_frames,
    )
    tracking_frames = count_frames(TRACKING_MASTER_CV, ENV_FFPROBE)

print(json.dumps(
    {
        "expected_frames": expected_frames,
        "tracking_frames": tracking_frames,
        "tracking_master_cv": str(TRACKING_MASTER_CV),
    },
    indent=2,
))

if tracking_frames != expected_frames:
    raise RuntimeError(f"Tracking master frame count mismatch: expected {expected_frames}, got {tracking_frames}")

if DELETE_INTERMEDIATES_AFTER_SUCCESS:
    for path in [
        CHUNK_IN_DIR,
        CHUNK_NORMAL_DIR,
        TMP_DIR,
        MEZZ_DIR,
        META_DIR,
        LOG_DIR,
        RUN_DIR / "chunks",
        RUN_DIR / "trimmed",
    ]:
        remove_path(path)

    for p in list(FINAL_DIR.iterdir()):
        if p.name != TRACKING_MASTER_CV.name:
            remove_path(p)

print("Done.")
print(f"Final file: {TRACKING_MASTER_CV}")